# Task 4 – End-to-End Machine Learning Project
## Student Pass/Fail Prediction

**Goal:** Predict whether a student is likely to pass based on information available before the final grade.

### Before running
1. Download `student-mat.csv` from the UCI Student Performance dataset.
2. Put it in the same folder as this notebook.
3. Run the notebook from top to bottom.

**Target definition:** Pass = final Maths grade (`G3`) is 10 or above; Fail = below 10.

**Leakage prevention:** `G1` and `G2` are excluded from the input features 
because they are earlier grades and may not be available in an early-warning prediction scenario.
 The final grade `G3` is used only to create the target.

In [17]:
# 1. IMPORT LIBRARIES
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
DATA_FILE = "student-mat.csv"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. LOAD THE DATASET
 
 Download the Maths dataset file named `student-mat.csv` 
from the UCI repository and place it next to this notebook.


In [18]:
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "student-mat.csv was not found. Download it from "
        "https://archive.ics.uci.edu/dataset/320/student+performance "
        "and place student-mat.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE, sep=";")
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (395, 1)


,"school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3"
0,"GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mot..."
1,"GP,F,17,U,GT3,T,1,1,at_home,other,course,fathe..."
2,"GP,F,15,U,LE3,T,1,1,at_home,other,other,mother..."
3,"GP,F,15,U,GT3,T,4,2,health,services,home,mothe..."
4,"GP,F,16,U,GT3,T,3,3,other,other,home,father,1,..."


3.UNDERSTAND AND CHECK THE DATA 

In [19]:
print("Columns:")
print(df.columns.tolist())
print("\\nData types:")
print(df.dtypes)
print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
print("\\nNumerical summary:")
display(df.describe())

Columns:
['school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3']
\nData types:
school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3    object
dtype: object
\nMissing values:


,missing_values
"school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3",0


Duplicate rows: 0
\nNumerical summary:


,"school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3"
count,395
unique,395
top,"GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mot..."
freq,1


## 4. CLEAN DATA AND CREATE THE TARGET


In [ ]:
df = df.drop_duplicates().copy()

# Remove rows without a final grade; G3 is needed to create the target.
df = df.dropna(subset=["G3"]).copy()

# Pass/fail label: 1 = Pass, 0 = Fail
df["Pass"] = (df["G3"] >= 10).astype(int)

print("Target counts:")
display(df["Pass"].value_counts().rename(index={0: "Fail", 1: "Pass"}))


## 5. EXPLORATORY DATA ANALYSIS

The following charts help understand the dataset and target distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.figure(figsize=(6, 4))
sns.countplot(data=df,x="Pass")
plt.title("Student Pass/Fail Distribution")
plt.xlabel("Pass (1) / Fail (0)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pass_fail_distribution.png"), dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="G3", bins=11, kde=True)
plt.title("Final Maths Grade Distribution")
plt.xlabel("Final grade (G3)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "final_grade_distribution.png"), dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Pass", y="absences")
plt.title("Absences by Pass/Fail Group")
plt.xlabel("Pass (1) / Fail (0)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "absences_by_result.png"), dpi=150)
plt.show()

## 6. PREPARE FEATURES AND TARGET
to remove `G1`, `G2`, `G3`, and the generated `Pass` column from model inputs. This avoids using grade information that would reveal or closely determine the final result.

In [ ]:
drop_columns = ["G1", "G2", "G3", "Pass"]
X = df.drop(columns=drop_columns)
y = df["Pass"]

feature_columns = X.columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of features:", len(feature_columns))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. BUILD PREPROCESSING AND MODEL PIPELINES

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

trained_models = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    trained_models[name] = pipe
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

comparison = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(comparison)
comparison.to_csv("model_comparison.csv", index=False)

## 8. COMPARE MODEL PERFORMANCE

In [ ]:

comparison_plot = comparison.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]]
comparison_plot.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150)
plt.show()

best_model_name = comparison.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Selected model:", best_model_name)

## 9. EVALUATE THE SELECTED MODEL

In [ ]:

y_pred = best_model.predict(X_test)

print("Classification report:")
print(classification_report(
    y_test, y_pred, target_names=["Fail", "Pass"], zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Fail", "Pass"], cmap="Blues"
)
plt.title(f"Confusion Matrix – {best_model_name}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 10. SAVE THE MODEL AND PROJECT OUTPUTS

In [ ]:
model_bundle = {
    "model": best_model,
    "feature_columns": feature_columns,
    "best_model_name": best_model_name,
    "pass_threshold": 10
}
joblib.dump(model_bundle, "best_student_model.joblib")

# Save cleaned dataset including the target label.
df.to_csv("student_performance_cleaned.csv", index=False)

# Save actual and predicted test labels.
prediction_results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})
prediction_results.to_csv("test_predictions.csv", index=False)

print("Saved:")
print("- best_student_model.joblib")
print("- model_comparison.csv")
print("- student_performance_cleaned.csv")
print("- test_predictions.csv")
print("- charts inside the outputs folder")

## 11. TRY A SAMPLE PREDICTION

In [ ]:
Sample_student = X_test.iloc[[0]].copy()
sample_prediction = best_model.predict(sample_student)[0]
print("Predicted result:", "PASS" if sample_prediction == 1 else "FAIL")
print("Actual result:", "PASS" if y_test.iloc[0] == 1 else "FAIL")

## 12. CONCLUSION

Write your conclusion after running the notebook. Mention:
- Which model achieved the highest F1-score.
- The test accuracy, precision, recall and F1-score.
- What the confusion matrix indicates.
- Limitations: this dataset is relatively small, and predictions are estimates, not guarantees about an individual student's future performance.

**Important:** Do not claim that the model guarantees a student's result. Use predictions as a learning-support signal only.